# Letter figure: N-extrapolation, d=2

3×3 grid: rows = rescale, columns = alpha.  
5 overlaid configs, linear fit bands, extrapolation to 1/N=0 with insets on α=−1 panels.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

data_dir = Path('data/scan_1e8')

d = 2
Lambda = 100
N_values = [30, 50, 100]
rescale_values = [0.3, 1.0, 3.0]
rescale_labels = [r'$r = 0.3\, m^{-1}$', r'$r = m^{-1}$', r'$r = 3\, m^{-1}$']
alphas = [-1, 0, 1]
M_str = '1e8'
n_configs_total = 50
n_show = 5

inv_N = np.array([1.0 / N for N in N_values])

## Helper functions

In [ ]:
def load_data(d, N, Lambda, alpha, rescale, M_str):
    r_str = f'{rescale:g}'
    fname = f'G4_d{d}_N{N}_L{Lambda:g}_a{alpha:g}_r{r_str}_M{M_str}.npz'
    data = np.load(data_dir / fname)
    return data['means'], data['errors'], data['theory_reg'], data['configs']


def linear_fit(inv_N, ratios, ratio_errs):
    w = 1.0 / ratio_errs**2
    W = np.sum(w)
    Wx = np.sum(w * inv_N)
    Wy = np.sum(w * ratios)
    Wxx = np.sum(w * inv_N**2)
    Wxy = np.sum(w * inv_N * ratios)
    det = W * Wxx - Wx**2
    a = (Wxx * Wy - Wx * Wxy) / det
    b = (W * Wxy - Wx * Wy) / det
    cov = np.array([[Wxx, -Wx], [-Wx, W]]) / det
    return a, b, cov


def fit_band(x, a, b, cov):
    y = a + b * x
    var_y = cov[0, 0] + 2 * x * cov[0, 1] + x**2 * cov[1, 1]
    y_err = np.sqrt(np.maximum(var_y, 0))
    return y, y_err

## Load data

In [ ]:
data_cache = {}
for alpha in alphas:
    for N in N_values:
        for rescale in rescale_values:
            means, errors, theory, configs = load_data(
                d, N, Lambda, alpha, rescale, M_str)
            data_cache[(N, alpha, rescale)] = (means, errors, theory, configs)

## Plotting helpers

In [ ]:
config_colors = plt.cm.Set2(np.linspace(0, 1, n_show))

x_pad = max(inv_N) * 0.12
x_max_band = max(inv_N) * 1.15
x_band = np.linspace(0, x_max_band, 80)
main_x_range = x_max_band + 1.5 * x_pad


def round_to_1sig(x):
    if x == 0:
        return 0
    exp = np.floor(np.log10(abs(x)))
    return round(x / 10**exp) * 10**exp


def compute_panel_data(alpha, rescale, config_indices):
    artists = []
    for color_idx, ci in enumerate(config_indices):
        y_vals, ye_vals = [], []
        for N in N_values:
            means, errors, theory, _ = data_cache[(N, alpha, rescale)]
            y_vals.append(means[ci] / theory[ci])
            ye_vals.append(abs(errors[ci] / theory[ci]))
        y = np.array(y_vals)
        ye = np.array(ye_vals)
        a, b, cov = linear_fit(inv_N, y, ye)
        artists.append((color_idx, y, ye, a, b, cov))
    return artists


def draw_panel(ax, artists):
    for ci, y, ye, a, b, cov in artists:
        color = config_colors[ci]
        y_b, y_be = fit_band(x_band, a, b, cov)
        ax.fill_between(x_band, y_b - y_be, y_b + y_be,
                        color=color, alpha=0.2)
        ax.plot(x_band, y_b, '-', color=color, alpha=0.5, linewidth=0.8)
        ax.errorbar(inv_N, y, yerr=ye, fmt='o', color=color,
                    markersize=3, capsize=1.5, linewidth=0.8)
        a_err = np.sqrt(cov[0, 0])
        ax.errorbar([0], [a], yerr=[a_err], fmt='o', color=color,
                    markersize=3.5, capsize=2,
                    markerfacecolor='none', markeredgewidth=1.0,
                    alpha=0.9)
    ax.axhline(1.0, color='k', linewidth=0.5, linestyle=':')


def auto_ylim(artists):
    all_y = []
    for ci, y, ye, a, b, cov in artists:
        all_y.extend(y + ye)
        all_y.extend(y - ye)
        a_err = np.sqrt(cov[0, 0])
        all_y.extend([a - a_err, a + a_err])
    y_min, y_max = min(all_y), max(all_y)
    margin = (y_max - y_min) * 0.08
    return (y_min - margin, y_max + margin)


def draw_inset(ax, artists, main_ylim):
    w_rel, h_rel = 0.35, 0.35
    iy = []
    for ci, y_d, ye_d, a, b, cov in artists:
        a_err = np.sqrt(cov[0, 0])
        iy.extend([a - a_err, a + a_err])
    iy_min, iy_max = min(iy), max(iy)
    spread = iy_max - iy_min
    pad = 0.1 * spread
    inset_y_range = 2 * (spread + 2 * pad)
    inset_y_lo = iy_min - pad
    inset_y_hi = inset_y_lo + inset_y_range
    main_y_range = main_ylim[1] - main_ylim[0]
    raw_x_range = inset_y_range * (main_x_range / main_y_range) * (w_rel / h_rel)
    inset_x_range = round_to_1sig(raw_x_range)
    x_ib = np.linspace(0, inset_x_range, 50)
    inset = ax.inset_axes([0.1, 0.6, w_rel, h_rel])
    for ci, y_d, ye_d, a, b, cov in artists:
        color = config_colors[ci]
        y_ib, y_ib_err = fit_band(x_ib, a, b, cov)
        inset.fill_between(x_ib, y_ib - y_ib_err, y_ib + y_ib_err,
                           color=color, alpha=0.2)
        inset.plot(x_ib, y_ib, '-', color=color, alpha=0.5, linewidth=0.8)
        a_err = np.sqrt(cov[0, 0])
        inset.errorbar([0], [a], yerr=[a_err], fmt='o', color=color,
                       markersize=2.5, capsize=1,
                       markerfacecolor='none', markeredgewidth=0.6, alpha=0.9)
    inset.axhline(1.0, color='k', linewidth=0.5, linestyle=':')
    inset.set_ylim(inset_y_lo, inset_y_hi)
    inset.set_xlim(-inset_x_range * 0.05, inset_x_range)
    inset.set_xticks([0, inset_x_range])
    inset.set_xticklabels(['0', f'{inset_x_range:.0e}'])
    inset.tick_params(labelsize=5)


def make_figure(config_indices):
    fig, axes = plt.subplots(3, 3, figsize=(10, 7.5))
    for row_idx, rescale in enumerate(rescale_values):
        for col_idx, alpha in enumerate(alphas):
            ax = axes[row_idx, col_idx]
            artists = compute_panel_data(alpha, rescale, config_indices)
            draw_panel(ax, artists)
            ax.set_xlim(-x_pad, x_max_band + x_pad * 0.5)
            ylim = auto_ylim(artists)
            ax.set_ylim(*ylim)
            ax.tick_params(labelsize=7)
            if alpha == -1:
                draw_inset(ax, artists, ylim)
            if row_idx == 0:
                ax.set_title(rf'$\alpha = {alpha}$', fontsize=11)
            if col_idx == 0:
                ax.set_ylabel(rescale_labels[row_idx] + '\n\nNNFT / Theory',
                              fontsize=9)
            if row_idx == 2:
                ax.set_xlabel(r'$1/N$', fontsize=9)
    fig.tight_layout()
    return fig

## Letter figure (configs 0–4)

The 3×3 grid shown in the paper, plotted from the first 5 configurations.

In [ ]:
fig = make_figure(range(n_show))
plt.show()

## Save the letter figure

Run this cell to write `n_extrapolation_d2.pdf` to disk. **Note:** this overwrites the reference PDF shipped in the repository.

In [ ]:
fig.savefig('n_extrapolation_d2.pdf', bbox_inches='tight')
print('Saved n_extrapolation_d2.pdf')

## All 50 configurations (multi-page PDF)

Generates a 10-page PDF (`n_extrapolation_d2_all_configs.pdf`), one page per batch of 5 consecutive configurations, to verify that the remaining 45 configurations exhibit the same qualitative behavior.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

n_batches = n_configs_total // n_show
out_pdf = 'n_extrapolation_d2_all_configs.pdf'
with PdfPages(out_pdf) as pdf:
    for batch in range(n_batches):
        idx_lo = batch * n_show
        idx_hi = idx_lo + n_show
        fig_b = make_figure(range(idx_lo, idx_hi))
        fig_b.suptitle(f'Configs {idx_lo}–{idx_hi - 1}', fontsize=10, y=1.02)
        pdf.savefig(fig_b, bbox_inches='tight')
        plt.close(fig_b)
print(f'Saved {out_pdf} ({n_batches} pages)')